# Nemotron 3 Nano 4B through NVIDIA NIM

*Local AI Inference Lab: Serving, Optimization and Application Quality*

**Question:** How do we make local AI faster and more efficient without losing useful task performance?

**Learn:** inspect NIM profiles, packaged serving and application request contracts.
**Requirements:** [CPU replay prerequisites](../SETUP.md); [Live configuration](../tutorial/RUN_ALL.md) only for fresh GPU runs.
**Expected outputs:** recorded tables, a server command/request/evaluated SQL example,
and an observation → explanation → limitation → decision for every experiment.

This optional extension applies the same quality-first method to packaged deployment. Read the Qwen/vLLM chapter first and use the [shared NIM technical guide](../tutorial/NIM_MVP4.md).

These are observations from a small recorded Dell workload, not universal claims. A faster
response is not necessarily a better answer; different optimizations affect different stages.
Clinical-trial analytics supplies an executable application-quality test. We have not evaluated
multiple application domains. The synthetic database contains no patient data.


## 1. Choose how to explore

| Mode | Behavior |
|---|---|
| **Replay — default** | Load checksum-verified **recorded Dell measurements** and reproduce tables without a GPU. |
| Plan | Show configuration and server commands without launching or downloading anything. |
| Live | Launch owned servers, benchmark, preserve evidence and clean up. Requires Linux GPU setup. |

Choose **Run All** now for replay. To change mode, create `tutorial/config/nim-nemotron.local.json`
using [the shared setup guide](../tutorial/RUN_ALL.md) and set `mode` to `replay`, `plan` or `live`.
All machine settings and flags belong in this configuration file. A live failure never silently
loads archived results. Replay ignores new experiment settings when presenting historical values.

Serving target: **Nemotron 3 Nano 4B**, `nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16`. The checkpoint is loaded as **FP16** through
a vLLM-backed NIM model-free container with a read-only local checkpoint mount.
The hybrid Mamba/attention architecture retains FP32 SSM state. Its BF16 source label does not change the requested FP16 experiment.

Four distinct clinical questions are repeated five times per serving condition. They are not
20 independent clinical tasks. Application quality separately evaluates four cases, including
planning, validation, SQL execution and one repair. Exact-SQL semantic review is still pending.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.workflow import Study
from clinical_trial_mvp.tutorial.teaching import server_example, request_example, response_example, interpret
if 'study' in globals():
    study.close()
study = Study('nim', 'nemotron', root=ROOT)
results = study.results
print('Mode:', study.mode)
print('Model:', 'nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16')
print(study.prepare())


Mode: replay
Model: nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16
Replay: recorded Dell measurements; no model download, server, GPU or network calls.


## 2. Predict before measuring

Recap: serving speed and application quality are different outcomes. Use
[chapter 01](01_qwen_cache_quantization.ipynb) for prefill/decode, W4A16 and common metric
definitions. Here, focus on the model/deployment differences below and predict their effects.


NIM packages an inference backend; it does not imply a TensorRT conversion.
Select your own runnable non-LoRA profile after model/image/GPU discovery. TP1/PP1 means one
GPU and one pipeline stage. Startup still has to succeed after a runnable profile listing.
The core study changes schema and client concurrency while reusing a fixed deployment.

The recorded Nemotron NIM runs used schema enforcement off; this remains its configurable default. Hybrid state and trusted local-code requirements must also be honored by the container backend.

Thinking parity with native vLLM remains unverified. The common `/no_think` prompt is not proof
of identical backend behavior. NIM versus native results are deployment observations until
model, request contracts, engine settings, date/database and thinking controls are matched.
Cache/AWQ extensions remain conditional; no new runtime is added to this lab.


## 3. Compatibility before benchmarking

Smoke asks whether the path can generate and execute SQL. It is not a general quality score.
Replay loads the recorded smoke when available. Plan makes no model calls. Live stops on
failed required smoke, retains the failure and never substitutes replay measurements.


In [2]:
smoke = study.smoke()
if isinstance(smoke, dict):
    print(smoke.get('_recorded', {}).get('label', study.mode), smoke.get('status'), smoke.get('reason', ''))
else:
    print(smoke)


recorded Dell measurement completed 


## 4. Experiments and decisions

Read each prediction before running its cell. Quality counts accompany speed; unavailable metrics stay unavailable.


### baseline


In [3]:
result = study.run('baseline')
print(result.get('_recorded', {}).get('label', study.mode), 'baseline:', result['status'])
display(Markdown(interpret(study, 'baseline')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'baseline')))


recorded Dell measurement baseline: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nim/runs/20261107T204614_919475Z.json](../experiment-tracker/mvp4/nim/runs/20261107T204614_919475Z.json)

**Prediction:** Establish executable SQL and separate application quality from request speed.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 8.889/12.469 s; aggregate output: 42.567 tok/s.

**Explanation:** Transport timing and application correctness measure different stages.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use this baseline to judge useful completed tasks, not latency alone.

### Under the hood: three inspectable paths

**A. Server command.** The next cell shows the recorded portable argv or the planned command,
model revision, dtype, cache control and scheduler limits. A setting absent from the archive
is marked unknown rather than inferred.
Raw Docker argv was not captured; the shown shape is reconstructed from metadata. [NIM deployment](../src/clinical_trial_mvp/tutorial/nim_study.py).


In [4]:
display(JSON(server_example(study, 'baseline')))


<IPython.core.display.JSON object>

**B. One HTTP request.** Inspect the full prompt, generation options and schema
contract. This uses the exact payload builder called by the transport. Replay reconstructs
it from the saved prompt and recorded flags; it is not a packet capture. No request is sent
by this inspection cell. [Request builder and timing](../src/clinical_trial_mvp/serving/client.py).


In [5]:
display(JSON(request_example(study, 'baseline')))


<IPython.core.display.JSON object>

**C. One evaluated response.** Inspect generated SQL, safety/schema validation,
execution outcome, query rows, the reference query and the recorded result comparison.
A reference match is not a human semantic review. [Plan/SQL validation](../src/clinical_trial_mvp/agent/analyst.py)
· [reference scorer](../src/clinical_trial_mvp/evaluation/scorers.py).


In [6]:
display(JSON(response_example(study, 'baseline')))


<IPython.core.display.JSON object>

### schema retrieved


In [7]:
result = study.run('schema_retrieved')
print(result.get('_recorded', {}).get('label', study.mode), 'schema_retrieved:', result['status'])
display(Markdown(interpret(study, 'schema_retrieved')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'schema_retrieved')))


recorded Dell measurement schema_retrieved: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nim/runs/20261107T205138_075077Z.json](../experiment-tracker/mvp4/nim/runs/20261107T205138_075077Z.json)

**Prediction:** Less context may reduce input work but can remove definitions required for correct SQL.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 5.768/10.374 s; aggregate output: 47.809 tok/s. Baseline application result match: 2/4.

**Explanation:** Context reduction changes the information available to the model, not just compute cost.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Retain needed clinical definitions; accept reduced context only with adequate task quality.

### concurrency 2


In [8]:
result = study.run('concurrency_2')
print(result.get('_recorded', {}).get('label', study.mode), 'concurrency_2:', result['status'])
display(Markdown(interpret(study, 'concurrency_2')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'concurrency_2')))


recorded Dell measurement concurrency_2: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nim/runs/20261107T205602_380125Z.json](../experiment-tracker/mvp4/nim/runs/20261107T205602_380125Z.json)

**Prediction:** More overlapping clients may improve aggregate throughput while increasing per-request latency.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 10.392/13.710 s; aggregate output: 73.331 tok/s.

**Explanation:** Client overlap and scheduler capacity are separate controls; these records do not show achieved batch occupancy.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Choose concurrency using both latency limits and correctly completed work.

### concurrency 4


In [9]:
result = study.run('concurrency_4')
print(result.get('_recorded', {}).get('label', study.mode), 'concurrency_4:', result['status'])
display(Markdown(interpret(study, 'concurrency_4')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'concurrency_4')))


recorded Dell measurement concurrency_4: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nim/runs/20261107T205940_074402Z.json](../experiment-tracker/mvp4/nim/runs/20261107T205940_074402Z.json)

**Prediction:** More overlapping clients may improve aggregate throughput while increasing per-request latency.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 12.044/22.403 s; aggregate output: 110.156 tok/s.

**Explanation:** Client overlap and scheduler capacity are separate controls; these records do not show achieved batch occupancy.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Choose concurrency using both latency limits and correctly completed work.

## 5. Put performance beside quality

TTFT is first generated content. Request E2E measures generation/transport; application E2E
includes planning, validation, execution and repair. Cold/warm cache phases are separate from
supplemental warmed summaries. Semantic review remains unreviewed. Native VRAM telemetry
failed and stays missing; no memory-reduction conclusion is supported by those runs.


In [10]:
display(Markdown(study.report()))
display(Markdown(study.cache_report()))


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| MVP4/NIM/nemotron/smoke · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2425 | 1.06 | 8.87/12.33 | 42.65 | 13.85 GiB |
| MVP4/NIM/nemotron/baseline · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2425 | 1.06 | 8.89/12.47 | 42.57 | 13.85 GiB |
| MVP4/NIM/nemotron/schema_retrieved · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 480 | 0.21 | 5.77/10.37 | 47.81 | 13.85 GiB |
| MVP4/NIM/nemotron/concurrency_2 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2425 | 1.11 | 10.39/13.71 | 73.33 | 13.85 GiB |
| MVP4/NIM/nemotron/concurrency_4 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2425 | 1.14 | 12.04/22.40 | 110.16 | 13.89 GiB |

| Experiment | Application E2E p50/p95 s | Request TTFT p95 s |
|---|---|---|
| smoke · recorded Dell measurement | 23.883/34.539 | 1.066 |
| baseline · recorded Dell measurement | 23.218/34.680 | 1.070 |
| schema_retrieved · recorded Dell measurement | 22.419/36.348 | 0.271 |
| concurrency_2 · recorded Dell measurement | 23.367/34.740 | 1.749 |
| concurrency_4 · recorded Dell measurement | 23.401/34.794 | 3.519 |

No controlled prefix-cache phase measurements in these results.

## 6. Finish, preserve and continue

Save with Ctrl+S. Replay changes only the notebook display; it does not rewrite measurements.
Live adds unique records and owns only the processes it starts.
Run the cleanup cell if pausing between successful live NIM cells.
Interrupted/failed operations clean up owned servers; a kernel/OS crash requires explicit
cleanup using the saved container ID.


In [11]:
print(study.close())


Study servers stopped. Saved records and notebook outputs are retained.


Continue to [the next chapter](05_edge_portability_executorch.ipynb), or read:

- [What should I choose?](../tutorial/RESULTS.md#what-should-i-choose) — concise results and deployment decisions.
- [Troubleshooting case studies](../tutorial/CASE_STUDIES.md).
- [Four-case SQL review packet](../tutorial/SQL_REVIEW.md) — human review pending.
- [Fresh GPU validation checklist](../tutorial/GPU_VALIDATION.md).
- [Original executed notebook](history/dell-2026-11-07/local%20AI%20optimization%20-%20NIM%20Nemotron.ipynb) and [checksum manifest](history/dell-2026-11-07/MANIFEST.json).

Historical iteration labels and source lineage are preserved in the evidence, not used as
the learner-facing chapter structure.
